# 🌦️ REFERENCE: KNMI Weather API Implementation

**Purpose**: Alternative weather data source using official Dutch meteorological data

**When to Use This**:
- Need official Netherlands weather records for production models
- Want authoritative source for regulatory/research purposes
- Require detailed local measurements from Dutch weather stations

**When to Use Open-Meteo Instead** (M2_02 notebook):
- Learning and experimentation (simpler API)
- International scope (multiple countries)
- Quick prototyping (no complex parsing needed)

---

## 📖 Overview

This reference notebook demonstrates how to fetch weather data from **KNMI (Koninklijk Nederlands Meteorologisch Instituut)**, the official Dutch meteorological institute.

**Key Differences from Open-Meteo**:
- ✅ **Authoritative**: Official Dutch government weather data
- ✅ **High Quality**: Calibrated instruments, quality-controlled data
- ✅ **Detailed**: More variables and station-specific measurements
- ⚠️ **Complex Format**: Returns fixed-width text format (not JSON)
- ⚠️ **Netherlands Only**: Limited to Dutch weather stations
- ⚠️ **More Parsing**: Requires extra code to clean and structure data

**Station 240 (Schiphol Airport)** is closest to Amsterdam and provides reliable measurements.

---

## Part 1: Understanding KNMI API Structure

### API Endpoint

**Base URL**: `https://www.daggegevens.knmi.nl/klimatologie/daggegevens`

**Parameters**:
- `stns` - Station ID(s), comma-separated (e.g., 240 for Schiphol)
- `start` - Start date in YYYYMMDD format (e.g., 20240101)
- `end` - End date in YYYYMMDD format (e.g., 20240131)

### Example Request

```
https://www.daggegevens.knmi.nl/klimatologie/daggegevens?stns=240&start=20240101&end=20240131
```

### Response Format

KNMI returns **fixed-width text format** with:
1. Comment lines starting with `#` (metadata, variable descriptions)
2. Header line with column names
3. Data lines with comma-separated values

**Key Variables**:
- `YYYYMMDD` - Date
- `TG` - Daily mean temperature (0.1°C)
- `TN` - Minimum temperature (0.1°C)
- `TX` - Maximum temperature (0.1°C)
- `RH` - Precipitation (0.1mm)
- `FG` - Daily mean wind speed (0.1 m/s)
- `DR` - Precipitation duration (0.1 hour)
- `SQ` - Sunshine duration (0.1 hour)

**Important**: All values use 0.1 units and need conversion!

---

## Part 2: Fetch Daily Weather Data

In [ ]:
# ═══════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════

import requests
import pandas as pd
from io import StringIO
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful!")

### Function: fetch_knmi_daily_data()

Complete implementation with error handling and data cleaning.

In [ ]:
def fetch_knmi_daily_data(station_id=240, start_date='20240101', end_date='20240131'):
    """
    Fetch daily weather data from KNMI API.
    
    Parameters:
    -----------
    station_id : int
        Station ID (240 = Schiphol Airport, near Amsterdam)
        Other options: 210 (Valkenburg), 260 (De Bilt)
    start_date : str
        Start date in YYYYMMDD format
    end_date : str
        End date in YYYYMMDD format
    
    Returns:
    --------
    pd.DataFrame or None
        Weather data with columns: date, temp_avg, temp_min, temp_max, 
        precipitation, wind_speed, sunshine_hours, etc.
        Returns None if request fails.
    
    Example:
    --------
    >>> df = fetch_knmi_daily_data(station_id=240, start_date='20240101', end_date='20240131')
    >>> print(df.head())
    """
    # KNMI API endpoint for daily data
    url = 'https://www.daggegevens.knmi.nl/klimatologie/daggegevens'
    
    params = {
        'stns': station_id,
        'start': start_date,
        'end': end_date
    }
    
    try:
        print(f"📡 Fetching KNMI data for station {station_id}...")
        print(f"   Date range: {start_date} to {end_date}")
        
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        
        # KNMI returns fixed-width text format with comment lines
        # Skip lines starting with '#'
        lines = [line for line in response.text.split('\n') 
                 if line.strip() and not line.strip().startswith('#')]
        
        if len(lines) < 2:
            print("❌ No data returned from KNMI")
            return None
        
        data_text = '\n'.join(lines)
        
        # Parse into DataFrame
        df = pd.read_csv(StringIO(data_text), skipinitialspace=True)
        
        # Convert date column (YYYYMMDD) to datetime
        df['date'] = pd.to_datetime(df['YYYYMMDD'].astype(str), format='%Y%m%d')
        
        # Select and rename key columns
        # All values are in 0.1 units and need conversion
        df_clean = pd.DataFrame({
            'date': df['date'],
            'station_id': df['STN'],
            'temp_avg': df['TG'] / 10,      # Convert to °C
            'temp_min': df['TN'] / 10,      # Convert to °C
            'temp_max': df['TX'] / 10,      # Convert to °C
            'precipitation': df['RH'] / 10,  # Convert to mm
            'wind_speed': df['FG'] / 10,    # Convert to m/s
            'sunshine_hours': df['SQ'] / 10 if 'SQ' in df.columns else None  # Convert to hours
        })
        
        # Remove any rows with missing dates
        df_clean = df_clean.dropna(subset=['date'])
        
        print(f"✅ Successfully fetched {len(df_clean)} days of data")
        print(f"   Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
        
        return df_clean
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error: {e}")
        return None
    except KeyError as e:
        print(f"❌ Missing expected column in KNMI response: {e}")
        return None
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        return None

# Test the function
print("\n" + "="*60)
print("Testing KNMI API...")
print("="*60 + "\n")

# Fetch last 30 days of data
end_date = datetime.now()
start_date = end_date - timedelta(days=30)

df_knmi = fetch_knmi_daily_data(
    station_id=240,
    start_date=start_date.strftime('%Y%m%d'),
    end_date=end_date.strftime('%Y%m%d')
)

if df_knmi is not None:
    print("\n📊 Sample Data:")
    print(df_knmi.head())
    
    print("\n📈 Summary Statistics:")
    print(df_knmi[['temp_avg', 'temp_min', 'temp_max', 'precipitation', 'wind_speed']].describe())

---

## Part 3: Compare with Open-Meteo

Let's fetch the same data from Open-Meteo and compare.

In [ ]:
def fetch_openmeteo_daily_data(latitude=52.3676, longitude=4.9041, start_date='2024-01-01', end_date='2024-01-31'):
    """
    Fetch daily weather data from Open-Meteo API (for comparison).
    
    Parameters:
    -----------
    latitude : float
        Latitude (52.3676 for Amsterdam)
    longitude : float
        Longitude (4.9041 for Amsterdam)
    start_date : str
        Start date in YYYY-MM-DD format
    end_date : str
        End date in YYYY-MM-DD format
    
    Returns:
    --------
    pd.DataFrame or None
        Weather data with comparable columns to KNMI
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'temperature_2m_mean,temperature_2m_min,temperature_2m_max,precipitation_sum,windspeed_10m_max',
        'timezone': 'Europe/Amsterdam'
    }
    
    try:
        print(f"📡 Fetching Open-Meteo data...")
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        daily = data['daily']
        
        df = pd.DataFrame({
            'date': pd.to_datetime(daily['time']),
            'temp_avg': daily['temperature_2m_mean'],
            'temp_min': daily['temperature_2m_min'],
            'temp_max': daily['temperature_2m_max'],
            'precipitation': daily['precipitation_sum'],
            'wind_speed': daily['windspeed_10m_max']
        })
        
        print(f"✅ Successfully fetched {len(df)} days of data")
        return df
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Fetch Open-Meteo data for comparison
if df_knmi is not None:
    start_str = df_knmi['date'].min().strftime('%Y-%m-%d')
    end_str = df_knmi['date'].max().strftime('%Y-%m-%d')
    
    df_openmeteo = fetch_openmeteo_daily_data(
        latitude=52.3088,  # Schiphol coordinates
        longitude=4.7639,
        start_date=start_str,
        end_date=end_str
    )
    
    if df_openmeteo is not None:
        print("\n📊 Open-Meteo Sample:")
        print(df_openmeteo.head())

### Comparison Analysis

In [ ]:
if df_knmi is not None and df_openmeteo is not None:
    # Merge on date for comparison
    comparison = df_knmi.merge(df_openmeteo, on='date', suffixes=('_knmi', '_openmeteo'))
    
    print("\n" + "="*60)
    print("📊 KNMI vs Open-Meteo Comparison")
    print("="*60)
    
    # Temperature comparison
    temp_diff = (comparison['temp_avg_knmi'] - comparison['temp_avg_openmeteo']).abs()
    print(f"\n🌡️ Temperature Differences:")
    print(f"   Mean absolute difference: {temp_diff.mean():.2f}°C")
    print(f"   Max difference: {temp_diff.max():.2f}°C")
    
    # Precipitation comparison
    precip_diff = (comparison['precipitation_knmi'] - comparison['precipitation_openmeteo']).abs()
    print(f"\n🌧️ Precipitation Differences:")
    print(f"   Mean absolute difference: {precip_diff.mean():.2f} mm")
    print(f"   Max difference: {precip_diff.max():.2f} mm")
    
    print("\n💡 Observations:")
    print("   - Small differences expected due to different measurement locations")
    print("   - KNMI: Schiphol Airport (official station)")
    print("   - Open-Meteo: Model-based interpolation")
    print("   - For Amsterdam projects, either source is acceptable")
    print("   - Use KNMI for official records, Open-Meteo for ease of use")

---

## Part 4: Visualize KNMI Data

In [ ]:
import matplotlib.pyplot as plt

if df_knmi is not None:
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    
    # Temperature plot
    axes[0].plot(df_knmi['date'], df_knmi['temp_avg'], label='Average', color='orange', linewidth=2)
    axes[0].fill_between(df_knmi['date'], df_knmi['temp_min'], df_knmi['temp_max'], 
                         alpha=0.3, color='orange', label='Min-Max Range')
    axes[0].set_ylabel('Temperature (°C)', fontsize=12)
    axes[0].set_title('KNMI Daily Temperature - Schiphol Airport', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Precipitation plot
    axes[1].bar(df_knmi['date'], df_knmi['precipitation'], color='steelblue', alpha=0.7)
    axes[1].set_ylabel('Precipitation (mm)', fontsize=12)
    axes[1].set_title('KNMI Daily Precipitation', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Wind speed plot
    axes[2].plot(df_knmi['date'], df_knmi['wind_speed'], color='green', linewidth=2)
    axes[2].set_ylabel('Wind Speed (m/s)', fontsize=12)
    axes[2].set_xlabel('Date', fontsize=12)
    axes[2].set_title('KNMI Daily Wind Speed', fontsize=14, fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ Visualizations complete!")

---

## Part 5: Production Tips

### When to Use KNMI vs Open-Meteo

| Criterion | KNMI ✅ | Open-Meteo ✅ |
|-----------|---------|---------------|
| **Official Records** | Yes - Government source | No - Model-based |
| **API Simplicity** | No - Complex format | Yes - JSON |
| **Global Coverage** | No - Netherlands only | Yes - Worldwide |
| **Free Access** | Yes | Yes |
| **API Key Required** | No | No |
| **Historical Depth** | Extensive (decades) | Good (1940-present) |
| **Update Frequency** | Daily | Hourly |
| **Best For** | Production models, research | Learning, prototyping |

### Recommendation

**For this course**: Use **Open-Meteo** (M2_02 notebook)
- Simpler API makes learning easier
- JSON format is easier to parse
- Good enough accuracy for learning projects

**For production Amsterdam bike models**: Consider **KNMI**
- Authoritative source for Netherlands
- Higher quality local measurements
- Required for regulatory/research use

**Best Practice**: Develop with Open-Meteo, validate with KNMI before production deployment.

---

## 📚 Additional Resources

- **KNMI Data Documentation**: https://www.knmi.nl/kennis-en-datacentrum/achtergrond/data-ophalen-vanuit-een-script
- **Station List**: https://www.knmi.nl/nederland-nu/klimatologie/daggegevens (select station map)
- **Variable Descriptions**: Included in API response comments (lines starting with #)

**Station 240 (Schiphol)** Details:
- Location: 52°18' N, 4°46' E
- Elevation: -3.4m below sea level
- Distance from Amsterdam center: ~15 km southwest
- Representative for Amsterdam weather conditions

---

## 🎯 Summary

**Key Takeaways**:
1. ✅ KNMI provides official Dutch weather data via free API
2. ✅ More complex than Open-Meteo (fixed-width text format)
3. ✅ Requires unit conversion (all values in 0.1 units)
4. ✅ Station 240 (Schiphol) is closest to Amsterdam
5. ✅ Good for production models requiring official records
6. ✅ Open-Meteo is recommended for learning (M2_02 notebook)

**Next Steps**:
- For learning: Continue with M2_02 (Open-Meteo)
- For production: Use this notebook as reference when needed
- For research: Consider using both sources for validation

---

**Module Navigation**: Return to [README.md](README.md) | Continue to [M2_03](M2_03_data_storage.ipynb)